# Ablation Studies

## Experiments:
1. **θ_BS dimension**: {8, 16, 32, 64, 128, 256}
2. **Site injection type**: FiLM vs concat vs add
3. **Number of pre-training BSs**: {2, 4, 6}
4. **Cold start analysis**: NMSE vs number of training samples

In [ ]:
import sys
sys.path.insert(0, '../..')

import copy
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from src.data.dataset import ChannelEstimationDataset
from src.models.estimator import create_model
from src.models.baselines import PlainEstimator
from src.training.trainer import train_local, evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
DATA_DIR = '../../data/channels'
PRETRAIN_BS = [0, 1, 2, 3, 4, 5]
TEST_BS_ID = 6
BATCH_SIZE = 32

## 1. θ_BS Dimension Ablation

In [ ]:
DIMS = [8, 16, 32, 64, 128, 256]
dim_results = {}

# Pre-train dataset
pretrain_ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=PRETRAIN_BS)
n = len(pretrain_ds)
n_val = int(n * 0.15)
train_ds, val_ds = torch.utils.data.random_split(pretrain_ds, [n - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# Test dataset
test_ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[TEST_BS_ID])
n_t = len(test_ds)
n_tv = int(n_t * 0.3)
test_train, test_val = torch.utils.data.random_split(test_ds, [n_t - n_tv, n_tv])
test_train_loader = DataLoader(test_train, batch_size=BATCH_SIZE, shuffle=True)
test_val_loader = DataLoader(test_val, batch_size=BATCH_SIZE)

for dim in DIMS:
    print(f'\n--- θ_BS dim = {dim} ---')
    # Pre-train
    model = create_model(site_integration='film', site_embed_dim=dim).to(device)
    train_local(model, train_loader, val_loader, epochs=80, device=device, verbose=False)
    
    # Adapt θ_BS only
    adapted = copy.deepcopy(model).to(device)
    adapted.site_embedding.reset()
    adapted.freeze_encoder()
    adapted.freeze_task_head()
    res = train_local(adapted, test_train_loader, test_val_loader, epochs=50, lr=1e-2, device=device, verbose=False)
    
    dim_results[dim] = 10 * np.log10(res['best_val'])
    print(f'  dim={dim}: NMSE = {dim_results[dim]:.2f} dB')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(DIMS, [dim_results[d] for d in DIMS], 'o-', linewidth=2, markersize=8)
ax.set_xlabel('θ_BS dimension')
ax.set_ylabel('NMSE (dB)')
ax.set_title('Effect of Site Embedding Dimension')
ax.set_xscale('log', base=2)
ax.set_xticks(DIMS)
ax.set_xticklabels(DIMS)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_dim.png', dpi=150)
plt.show()

## 2. Site Injection Type

In [ ]:
INJECTION_TYPES = ['film', 'concat', 'add', 'none']
inject_results = {}

for inject_type in INJECTION_TYPES:
    print(f'\n--- Injection: {inject_type} ---')
    model = create_model(site_integration=inject_type, site_embed_dim=64).to(device)
    train_local(model, train_loader, val_loader, epochs=80, device=device, verbose=False)
    
    if inject_type != 'none':
        adapted = copy.deepcopy(model).to(device)
        adapted.site_embedding.reset()
        adapted.freeze_encoder()
        adapted.freeze_task_head()
        res = train_local(adapted, test_train_loader, test_val_loader, epochs=50, lr=1e-2, device=device, verbose=False)
    else:
        # For 'none': fine-tune all
        adapted = copy.deepcopy(model).to(device)
        res = train_local(adapted, test_train_loader, test_val_loader, epochs=50, lr=1e-4, device=device, verbose=False)
    
    inject_results[inject_type] = 10 * np.log10(res['best_val'])
    print(f'  {inject_type}: NMSE = {inject_results[inject_type]:.2f} dB')

print('\nResults:', inject_results)

## 3. Cold Start Analysis

In [ ]:
# How performance changes as target BS accumulates more data
# Compare: pre-trained + θ_BS adaptation vs from-scratch

SAMPLE_COUNTS = [5, 10, 20, 50, 100, 200, 500]
N_REPEATS = 3

cold_start = {'θ_BS adapt': {}, 'From scratch': {}}

# Pre-train model once
pretrained = create_model(site_integration='film', site_embed_dim=64).to(device)
train_local(pretrained, train_loader, val_loader, epochs=80, device=device, verbose=False)

test_indices = list(range(len(test_ds)))
np.random.shuffle(test_indices)
fixed_val_idx = test_indices[:int(len(test_ds) * 0.3)]
available_train_idx = test_indices[int(len(test_ds) * 0.3):]
fixed_val_loader = DataLoader(Subset(test_ds, fixed_val_idx), batch_size=BATCH_SIZE)

for n_samples in SAMPLE_COUNTS:
    if n_samples > len(available_train_idx):
        break
    
    adapt_scores = []
    scratch_scores = []
    
    for rep in range(N_REPEATS):
        sub_idx = np.random.choice(available_train_idx, n_samples, replace=False)
        sub_loader = DataLoader(Subset(test_ds, sub_idx), batch_size=min(n_samples, BATCH_SIZE), shuffle=True)
        
        # θ_BS adapt
        m1 = copy.deepcopy(pretrained).to(device)
        m1.site_embedding.reset()
        m1.freeze_encoder()
        m1.freeze_task_head()
        train_local(m1, sub_loader, epochs=50, lr=1e-2, device=device, verbose=False)
        _, db = evaluate(m1, fixed_val_loader, device)
        adapt_scores.append(db)
        
        # From scratch
        m2 = create_model(site_integration='film', site_embed_dim=64).to(device)
        train_local(m2, sub_loader, epochs=100, lr=1e-3, device=device, verbose=False)
        _, db = evaluate(m2, fixed_val_loader, device)
        scratch_scores.append(db)
    
    cold_start['θ_BS adapt'][n_samples] = (np.mean(adapt_scores), np.std(adapt_scores))
    cold_start['From scratch'][n_samples] = (np.mean(scratch_scores), np.std(scratch_scores))
    print(f'n={n_samples}: adapt={np.mean(adapt_scores):.2f}±{np.std(adapt_scores):.2f}, '
          f'scratch={np.mean(scratch_scores):.2f}±{np.std(scratch_scores):.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for method, color in zip(cold_start.keys(), ['C0', 'C2']):
    counts = sorted(cold_start[method].keys())
    means = [cold_start[method][n][0] for n in counts]
    stds = [cold_start[method][n][1] for n in counts]
    ax.errorbar(counts, means, yerr=stds, marker='o', label=method,
                color=color, capsize=3, linewidth=2)

ax.set_xlabel('Number of training samples')
ax.set_ylabel('NMSE (dB)')
ax.set_title(f'Cold Start: Adaptation Speed for BS{TEST_BS_ID}')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Find crossover point
counts = sorted(cold_start['θ_BS adapt'].keys())
adapt_means = [cold_start['θ_BS adapt'][n][0] for n in counts]
scratch_means = [cold_start['From scratch'][n][0] for n in counts]
for i, n in enumerate(counts):
    if scratch_means[i] <= adapt_means[i]:
        ax.axvline(x=n, color='gray', linestyle='--', alpha=0.5)
        ax.annotate(f'Crossover ~{n} samples', xy=(n, adapt_means[i]),
                   fontsize=9, color='gray')
        break

plt.tight_layout()
plt.savefig('ablation_cold_start.png', dpi=150)
plt.show()